# HW3P2: Automatic Speech Recognition with an Encoder-Decoder Transformer

# Schedule:
- Kaggle Submission (DUE 6 November 2026 @ 11:59PM EST | Slack Deadline is 13 November 2026 @ 11:59PM EST)
- Code Submission (DUE 8 November 2026 @ 11:59PM EST | Closes 13 November 2026 @ 11:59PM EST, with the Slack Kaggle Competition)


## Requirement Acknowledgement

Setting the flag below to **True** indicates that you have read, understood, and agree to comply with **all** of the following requirements:

1. **Slack Days Usage**  
   Slack days may be used **only** for the **P2 Final submission**.  
   Specifically, slack days may be applied to submit **final P2 Kaggle scores** to the **Slack Kaggle Competition**, at the expense of your available slack days.

2. **Final Code Submission Deadline**  
   The final **Gradescope code submission** is due **within 48 hours after** the Kaggle competition deadline.

3. **Kaggle Username & Score Verification**  
   You must provide your **Kaggle username** below. We will use it to retrieve your official **PRIVATE leaderboard** score.  
   Minor score or output variance is acceptable; however, any discrepancies caused by **modifications to the required submission code**
   (including the notebook’s final submission cell) will result in an **Academic Integrity Violation (AIV)**.

4. **Model Implementation Restrictions**  
   You are **not permitted** to use pre-trained or pre-loaded models (e.g., from Hugging Face or similar libraries).  
   You **may** implement models described in research papers or articles, but they **must** be implemented **from scratch**
   using fundamental PyTorch components (e.g., `Linear`, `Conv2d`, etc.).

5. **Data Usage Restrictions**  
   The use of **any external data or datasets** is strictly prohibited at **any stage** of this assignment.

6. **Collaboration Policy**  
   You may collaborate with teammates to run experiments or ablations.  
   However, you must submit **your own code** and **your own results**.

7. **Academic Integrity**  
   Failure to comply with any of the above requirements will be treated as an **Academic Integrity Violation (AIV)**.

8. **Late Submissions**  
   Late submissions **must** be submitted through the **Slack Kaggle Competition** (see write-up for details).  
   Submissions made to the **regular Kaggle competition** after the original deadline will **not** be considered,
   regardless of remaining slack days.
   
   Gradescope code submission will close at **the same time** as the Slack Kaggle Competition, rather than remaining open for an additional 48 hours.


In [14]:
ACKNOWLEDGED = True #TODO: Only set Acknowledged to True if you have read the above acknowlegements and agree to ALL of them.

# Setup
-  Follow the setup instructions based on your preferred environment!

Click [**HERE**](https://www.kaggle.com/t/6aedc76c79fa415fb477f967ffd7c799) to join the HW3P2 Kaggle competition first!

## Local

One of our key goals in designing this assignment is to allow you to complete most of the preliminary implementation work locally.  
We highly recommend that you **pass all tests locally** using the provided `hw3_data_subset` before moving to a GPU runtime.  
To do this, simply:

### Create a new conda environment
```bash
# Be sure to deactivate any active environments first
conda create -n hw3 python=3.12.4
```

### Activate the conda environment
```bash
conda activate hw3
```

### Install the dependencies using the provided `requirements.txt`
```bash
pip install --no-cache-dir --ignore-installed -r requirements.txt
```

### Ensure that your notebook is in the same working directory as the `Handout`
This can be achieved by:
1. Physically moving the notebook into the handout directory.
2. Changing the notebook’s current working directory to the handout directory using the os.chdir() function.

### Open the notebook and select the newly created environment from the kernel selector.

If everything was done correctly, You should see atleast the following files in your current working directory after running `!ls`:
```
.
├── README.md
├── requirements.txt
├── hw3lib/
├── tests/
└── hw3_data_subset/
```

## Kaggle

While it is possible to run the notebook on Kaggle, we would recommend against it. This assignment is more resource intensive and may run slower on Kaggle.

- `NOTE`: Turn on `Internet` in the notebook settings. Getting your handout and installing the dependencies both need it.

### Step 1: Get your handout
- See writeup for recommended approaches.

In [15]:
# Example: My preferred approach
import os
# Settings -> Developer Settings -> Personal Access Tokens -> Token (classic)
os.environ['GITHUB_TOKEN'] = "REDACTED"

GITHUB_USERNAME = "JingerBread9"
REPO_NAME       = "idl-hw3p2" # TODO: Your repo's name. The cells below use this variable, so set it here only
TOKEN = os.environ.get("GITHUB_TOKEN")
repo_url        = f"https://{TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
!git clone {repo_url}

Cloning into 'idl-hw3p2'...
remote: Enumerating objects: 198, done.
remote: Counting objects: 100% (198/198), done.
remote: Compressing objects: 100% (157/157), done.
remote: Total 198 (delta 58), reused 178 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (198/198), 13.55 MiB | 32.65 MiB/s, done.
Resolving deltas: 100% (58/58), done.


In [16]:
# To pull latest changes (Must be in the repo dir, use pwd/ls to verify)
!cd {REPO_NAME} && git pull

Already up to date.


### Step 2: Install Dependencies

In [17]:
!pip install wandb==0.23.1 torchinfo torchmetrics kaggle==1.8.0 "kagglehub>=1.0.0" --quiet

In [18]:
import wandb, torch, torchinfo, torchmetrics
print(wandb.__version__, torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

0.23.1 2.11.0+cu130 True Tesla T4


### Step 3: Get Data

If you are using Kaggle, do not download the data manually. Follow these steps to add the dataset directly to your notebook:
1. Open your **Kaggle Notebook**.
2. Navigate to **Notebook → Input**.
3. Click **Add Input**.
4. Search for the dataset `cmu11785/f26-hw3p2-data` (https://www.kaggle.com/datasets/cmu11785/f26-hw3p2-data).
5. Click the **➕ (plus sign)** to add the dataset to your notebook.

Then run the cell below. It prints where Kaggle put the data; use that path for `root` in the config file.

In [19]:
import glob
# The handout ships a small hw3_data_subset/hw3p2_data of its own, which is not the dataset
print([p for p in glob.glob('/kaggle/input/**/hw3p2_data', recursive=True) if 'hw3_data_subset' not in p])

[]


### Step 4: Configure Kaggle API Access

⚠️ **Important:** You must provide your own Kaggle API credentials while creating your deliverables for code submission and (optionally) submitting to the competition.

#### How to obtain your Kaggle API credentials

1. Navigate to your **Kaggle Profile → Settings** tab.
2. Click **“Generate New Token”**
3. Copy the token and paste it below. It is `KGAT_` followed by 32 letters and digits, for example `KGAT_0123456789abcdef0123456789abcdef`. The older 32-character key inside a `kaggle.json` file is a different credential and does not work here.
4. Set `KAGGLE_USERNAME` to your Kaggle username, as shown on your profile page, not your email address.

Join the HW3P2 kaggle competition (https://www.kaggle.com/t/6aedc76c79fa415fb477f967ffd7c799)

In [20]:
import os
import pprint
os.environ['KAGGLE_USERNAME'] = "hojunjin999" # TODO: Verify in Settings
os.environ['KAGGLE_API_TOKEN'] = "REDACTED" # TODO: Add your API token, KGAT_ followed by 32 letters and digits

# The new API token starts with KGAT_. The 32-character key in an older kaggle.json does not work here.
os.environ['KAGGLE_API_TOKEN'] = os.environ['KAGGLE_API_TOKEN'].strip()
assert os.environ['KAGGLE_API_TOKEN'].startswith("KGAT_"), "Use the API token that starts with KGAT_ (Kaggle Profile → Settings → Generate New Token)"

# Verify
import kaggle
api = kaggle.api  # Already authenticated on import
api.competitions_list_cli()

ref                                                                                deadline             category         reward  teamCount  userHasEntered  
---------------------------------------------------------------------------------  -------------------  --------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/passenger-screening-algorithm-challenge        2017-12-15 23:59:00  Featured  1,500,000 Usd        518           False  
https://www.kaggle.com/competitions/zillow-prize-1                                 2018-01-10 15:59:00  Featured  1,200,000 Usd       3770           False  
https://www.kaggle.com/competitions/data-science-bowl-2017                         2017-04-12 23:59:00  Featured  1,000,000 Usd       1972           False  
https://www.kaggle.com/competitions/vesuvius-challenge-ink-detection               2023-06-14 23:59:00  Featured  1,000,000 Usd       1249           False  
https://www.kaggle.com/competitions/gemini-3              

In [23]:
import os
os.environ['KAGGLE_USERNAME']  = "hojunjin999"
os.environ['KAGGLE_API_TOKEN'] = "KGAT_put_your_token_here"
if not os.path.isdir("/tmp/data/hw3p2-data/hw3p2_data"):
    !kaggle datasets download -d cmu11785/f26-hw3p2-data -p /tmp/data --unzip
print(os.listdir("/tmp/data/hw3p2-data/hw3p2_data"))

Dataset URL: https://www.kaggle.com/datasets/cmu11785/f26-hw3p2-data
License(s): unknown
100%|██████████████████████████████████████▊| 11.8G/11.8G [00:50<00:00, 344MB/s]
100%|███████████████████████████████████████| 11.8G/11.8G [00:50<00:00, 252MB/s]
['test-clean', 'train-clean-100', 'dev-clean', 'char_set.txt']


### Step 5: Move to Handout Directory
You must be within the handout directory for the library imports to work!

- `NOTE`: You may have to repeat running this command anytime you restart your session.
- `NOTE`: You can do a `pwd` to check if you are in the right directory.

In [21]:
!grep -o 'HW3P2[^"\\]*' /tmp/ckpt/hw3p2.log | sort -u | head

HW3P2|WB=hojunj-carnegie-mellon-university/hw3p2/n3ocb7me


In [22]:
import os
os.chdir("/kaggle/working/idl-hw3p2")  # Set when you cloned the repo. After a restart, run that cell again first
!git pull
!git log --oneline -1

Already up to date.
32ed053 (HEAD -> main, origin/main, origin/HEAD) Use beam search in recognize


In [42]:
   ckpt = "/tmp/ckpt/idl-hw3p2/expts/add_run_name/checkpoints/checkpoint-last-epoch-model.pth"
   trainer.load_checkpoint(ckpt)

Checkpoint loaded from epoch 23
Successfully loaded: model, scaler, training_state


# Imports
- If your setup was done correctly, you should be able to run the following cell without any issues.

In [24]:
from hw3lib.data import (
    H3Tokenizer,
    ASRDataset,
    verify_dataloader
)
from hw3lib.model import (
    EncoderDecoderTransformer
)
from hw3lib.utils import (
    create_scheduler,
    create_optimizer,
    plot_lr_schedule
)
from hw3lib.trainers import (
    ASRTrainer,
    ProgressiveTrainer
)
from torch.utils.data import DataLoader
import yaml
import gc
import torch
from torchinfo import summary
import os
import json
import wandb
import pandas as pd
device = "cuda" if torch.cuda.is_available() else "cpu"


# Implementations
- `NOTE`: All of these implementations have detailed specification, implementation details, and hints in their respective source files. Make sure to read all of them in their entirety to understand the implementation details!

## Dataset Implementation
- Implement the `ASRDataset` class in `hw3lib/data/asr_dataset.py`.
- You will have to implement parts of `__init__` and completely implement the `__len__`, `__getitem__` and `collate_fn` methods.
- Run the cell below to check your implementation.


In [ ]:
!python -m tests.test_dataset_asr

## Model Implementations

Overview:

- Implement the `PadMask` and `CausalMask` functions in `hw3lib/model/masks.py`.
- Implement the `PositionalEncoding` and `RotaryPositionalEncoding` classes in `hw3lib/model/positional_encoding.py`.
- Implement the `RMSNorm` class in `hw3lib/model/norms.py`.
- Implement the `MultiHeadAttention` class in `hw3lib/model/attention.py`.
- Implement the `SelfAttentionLayer`, `CrossAttentionLayer`, `SwiGLU` and `FeedForwardLayer` classes in `hw3lib/model/sublayers.py`.
- Implement the `CrossAttentionDecoderLayer` class in `hw3lib/model/decoder_layers.py`.
- Implement the `SelfAttentionEncoderLayer` class in `hw3lib/model/encoder_layers.py`. This will be mostly a copy-paste of the `CrossAttentionDecoderLayer`, without the cross-attention sublayer and without the causal constraint on the input sequence.
- Implement the `EncoderDecoderTransformer` class in `hw3lib/model/transformers.py`.

Implement them in the order below. Each one is used by the next.

### Masks
- Implement the `PadMask` and `CausalMask` functions in `hw3lib/model/masks.py`.
- Run the cells below to check your implementations.

In [ ]:
!python -m tests.test_mask_padding

In [ ]:
!python -m tests.test_mask_causal

### Positional Encoding
- Implement the `PositionalEncoding` class in `hw3lib/model/positional_encoding.py`.
- This is the fixed sinusoidal encoding, added to the embeddings before the first layer.
- Run the cell below to check your implementation.

In [ ]:
!python -m tests.test_positional_encoding

### Rotary Positional Encoding
- Implement the `RotaryPositionalEncoding` class in `hw3lib/model/positional_encoding.py`.
- Unlike `PositionalEncoding`, this one is not added to the embeddings. It rotates the queries and keys inside each attention operation, so it is applied in `MultiHeadAttention` rather than at the top of the model.
- Run the cell below to check your implementation.

In [ ]:
!python -m tests.test_rope

### Normalization
- Implement the `RMSNorm` class in `hw3lib/model/norms.py`.
- Run the cell below to check your implementation.

In [ ]:
!python -m tests.test_norms

### Multi-Head Attention
- Implement the `MultiHeadAttention` class in `hw3lib/model/attention.py`.
- You implemented this in NumPy for `HW3P1`, in `mytorch/nn/multi_head_attention.py`. This is the same module in PyTorch, so keep that file open beside this one. The method names and the `forward` signature are unchanged, `backward` and `init_weights` are no longer needed, and rotary embeddings and QK-Norm are new.
- Run the cell below to check your implementation.

In [ ]:
!python -m tests.test_attention

### Transformer Sublayers
- Implement the `SelfAttentionLayer`, `CrossAttentionLayer`, `SwiGLU` and `FeedForwardLayer` classes in `hw3lib/model/sublayers.py`.
- `NOTE`: `SwiGLU` and `FeedForwardLayer` are separate. `FeedForwardLayer` selects between `SwiGLU` and the GELU network based on `ffn_type`.
- Run the cells below to check your implementations.

In [ ]:
!python -m tests.test_sublayer_selfattention

In [ ]:
!python -m tests.test_sublayer_feedforward

In [ ]:
!python -m tests.test_sublayer_crossattention

### Transformer Cross-Attention Decoder Layer
- Implement the `CrossAttentionDecoderLayer` class in `hw3lib/model/decoder_layers.py`.
- Then run the cell below to check your implementation.


In [ ]:
!python -m tests.test_decoderlayer_crossattention

### Transformer Self-Attention Encoder Layer
- Implement the `SelfAttentionEncoderLayer` class in `hw3lib/model/encoder_layers.py`.
- Then run the cell below to check your implementation.




In [ ]:
!python -m tests.test_encoderlayer_selfattention

### Encoder-Decoder Transformer

- Implement the  `EncoderDecoderTransformer` class in `hw3lib/model/transformers.py`.
- Then run the cell below to check your implementation.

In [ ]:
!python -m tests.test_transformer_encoder_decoder

## Decoding Implementation
- Implement the `generate_greedy` method of the `SequenceGenerator` class in `hw3lib/decoding/sequence_generator.py`.
- Greedy search is required. The trainer's `recognize` method uses it by default, and so does the inference cell at the end of this notebook.
- Run the cell below to check your implementation.

In [ ]:
!python -m tests.test_decoding --mode greedy

### Beam Search
- We highly recommend you to implement the `generate_beam` method of the `SequenceGenerator` class in `hw3lib/decoding/sequence_generator.py`.
- Then run the cell below to check your implementation.
- `NOTE`: This is an optional but highly recommended task for `HW3P2` to ease the journey to high cutoffs!

In [25]:
!python -m tests.test_decoding --mode beam


Running tests for category: Decoding
--------------------------------------------------------------------------------

[01/01]    Running:  Test beam decoding
Testing Single Batch Beam Search ...
Beam 0  : generated: HELLO WORLD  | expected: HELLO WORLD 
Beam 1  : generated: YELLOW WORLD | expected: YELLOW WORLD
Beam 2  : generated: MELLOW WORLD | expected: MELLOW WORLD
Testing Multi Batch Beam Search ...
Batch 0  : Beam 0  : generated: HELLO WORLD  | expected: HELLO WORLD 
Batch 0  : Beam 1  : generated: YELLOW WORLD | expected: YELLOW WORLD
Batch 0  : Beam 2  : generated: MELLOW WORLD | expected: MELLOW WORLD
Batch 1  : Beam 0  : generated: GOOD BYE     | expected: GOOD BYE    
Batch 1  : Beam 1  : generated: GREAT DAY    | expected: GREAT DAY   
Batch 1  : Beam 2  : generated: GUD NIGHT    | expected: GUD NIGHT   
[01/01]    PASSED:   Test beam decoding


                                  Test Summary                                  
Category:    Decoding                      
Res

## Trainer Implementation
You will have to do some minor in-filling for the `ASRTrainer` class in `hw3lib/trainers/asr_trainer.py` before you can use it.
- Fill in the `TODO`s in the `__init__`.
- Fill in the `TODO`s in the `_train_epoch`.
- Fill in the `TODO`s in the `recognize` method.
- Fill in the `TODO`s in the `_validate_epoch`.
- Fill in the `TODO`s in the `train` method.

`WARNING`: There are no test's for this. Implement carefully!

# Experiments
From this point onwards you may want to switch to a `GPU` runtime.
- `OBJECTIVE`: Optimize your model for `CER` on the test set.

## Config
- You can use the `config.yaml` file to set your config for your ablation study.

---
### Notes:

- Set `tokenization: token_type:` to specify your desired tokenization strategy
- You will need to set the root path to your `hw3p2_data` folder in `data: root:`. This will depend on your setup. For eg.
  - `Local`: `"hw3_data_subset/hw3p2_data"` (the small subset that ships with the handout)
  - `Colab`: `"/content/hw3p2-data/hw3p2_data"`
  - `Kaggle`: the path printed in Kaggle Step 3, for example `"/kaggle/input/datasets/cmu11785/f26-hw3p2-data/hw3p2-data/hw3p2_data"`
  - `PSC`: `"/local/hw3p2-data/hw3p2_data"`
- There's extra configurations in the `optimizer` section which will only be relevant if you decide to use the `create_optimizer` function we've provided in `hw3lib/utils/create_optimizer.py`.
- `BE CAREFUL` while setting numeric values. Eg. `1e-4` will get serialized to a `str` while `1.0e-4` gets serialized to float.

In [26]:
%%writefile config.yaml

Name                      : "Hojun Jin"

###### Tokenization ------------------------------------------------------------
tokenization:
  token_type                : "1k"       # [char, 1k, 5k, 10k]
  token_map :
      'char': 'hw3lib/data/tokenizer_jsons/tokenizer_char.json'
      '1k'  : 'hw3lib/data/tokenizer_jsons/tokenizer_1000.json'
      '5k'  : 'hw3lib/data/tokenizer_jsons/tokenizer_5000.json'
      '10k' : 'hw3lib/data/tokenizer_jsons/tokenizer_10000.json'

###### Dataset -----------------------------------------------------------------
data:
  root                 : "/tmp/data/hw3p2-data/hw3p2_data"
  train_partition      : "train-clean-100"
  val_partition        : "dev-clean"
  test_partition       : "test-clean"
  subset               : 1.0
  batch_size           : 16
  NUM_WORKERS          : 4
  norm                 : 'global_mvn'
  num_feats            : 80

  specaug                   : True
  specaug_conf:
    apply_freq_mask         : True
    freq_mask_width_range   : 5
    num_freq_mask           : 2
    apply_time_mask         : True
    time_mask_width_range   : 40
    num_time_mask           : 2

###### Network Specs -------------------------------------------------------------
model:
  input_dim: 80
  time_reduction: 4
  reduction_method: 'conv'

  d_model: 320
  num_encoder_layers: 8
  num_decoder_layers: 4
  num_encoder_heads: 8
  num_decoder_heads: 8
  d_ff_encoder: 1280
  d_ff_decoder: 1280
  encoder_pos_encoding: 'rope'
  decoder_pos_encoding: 'rope'
  norm_type: 'rmsnorm'
  ffn_type: 'swiglu'
  qk_norm: False

  dropout: 0.1
  layer_drop_rate: 0.0
  weight_tying: True

###### Common Training Parameters ------------------------------------------------
training:
  use_wandb                   : True
  wandb_run_id                : "none"
  gradient_accumulation_steps : 1
  wandb_project               : "hw3p2"

###### Loss ----------------------------------------------------------------------
loss:
  label_smoothing: 0.1
  ctc_weight: 0.3

###### Optimizer -----------------------------------------------------------------
optimizer:
  name: "adamw"
  lr: 0.0005
  weight_decay: 0.01

  param_groups:
    - name: self_attn
      patterns: []
      lr: 0.0002
      layer_decay:
        enabled: False
        decay_rate: 0.8

    - name: ffn
      patterns: []
      lr: 0.0002
      layer_decay:
        enabled: False
        decay_rate: 0.8

  layer_decay:
    enabled: False
    decay_rate: 0.75

  sgd:
    momentum: 0.9
    nesterov: True
    dampening: 0

  adam:
    betas: [0.9, 0.999]
    eps: 1.0e-8
    amsgrad: False

  adamw:
    betas: [0.9, 0.999]
    eps: 1.0e-8
    amsgrad: False

###### Scheduler -----------------------------------------------------------------
scheduler:
  name: "cosine"

  reduce_lr:
    mode: "min"
    factor: 0.1
    patience: 10
    threshold: 0.0001
    threshold_mode: "rel"
    cooldown: 0
    min_lr: 0.0000001
    eps: 1.0e-8

  cosine:
    T_max: 21        # 24 total epochs - 3 warmup
    eta_min: 0.0000001
    last_epoch: -1

  cosine_warm:
    T_0: 10
    T_mult: 10
    eta_min: 0.0000001
    last_epoch: -1

  warmup:
    enabled: True
    type: "linear"
    epochs: 3
    start_factor: 0.1
    end_factor: 1.0

Writing config.yaml


In [27]:
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)

## Tokenizer

In [28]:
Tokenizer = H3Tokenizer(
    token_map  = config['tokenization']['token_map'],
    token_type = config['tokenization']['token_type']
)

                          Tokenizer Configuration (1k)                          
--------------------------------------------------------------------------------
Vocabulary size:     1000

Special Tokens:
PAD:              0
UNK:              1
MASK:             2
SOS:              3
EOS:              4
BLANK:            5

Validation Example:
--------------------------------------------------------------------------------
Input text:  [SOS]HI DEEP LEARNERS[EOS]
Tokens:      ['[SOS]', 'H', 'I', 'ĠDE', 'EP', 'ĠLE', 'AR', 'N', 'ERS', '[EOS]']
Token IDs:   [3, 14, 15, 159, 290, 228, 71, 20, 214, 4]
Decoded:     [SOS]HI DEEP LEARNERS[EOS]


## Datasets

In [29]:
   import os
   from kaggle_secrets import UserSecretsClient
   os.environ['KAGGLE_USERNAME']  = "hojunjin999"
   os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN").strip()
   if not os.path.isdir("/tmp/data/hw3p2-data/hw3p2_data"):
       !kaggle datasets download -d cmu11785/f26-hw3p2-data -p /tmp/data --unzip
   print(os.listdir("/tmp/data/hw3p2-data/hw3p2_data"))

['test-clean', 'train-clean-100', 'dev-clean', 'char_set.txt']


## Dataloaders

In [30]:
!free -g
!df -h /tmp /kaggle/working

               total        used        free      shared  buff/cache   available
Mem:              31           2          17           0          12          29
Swap:              0           0           0
Filesystem      Size  Used Avail Use% Mounted on
overlay         8.0T  7.1T  951G  89% /
/dev/loop2       20G   32M   20G   1% /kaggle/working


In [31]:
train_dataset = ASRDataset(
    partition=config['data']['train_partition'],
    config=config['data'],
    tokenizer=Tokenizer,
    isTrainPartition=True,
    global_stats=None  # Will compute stats from training data
)

# Get the computed global stats from the training set
global_stats = None
if config['data']['norm'] == 'global_mvn':
    global_stats = (train_dataset.global_mean, train_dataset.global_std)
    print(f"Global stats computed from training set.")

val_dataset = ASRDataset(
    partition=config['data']['val_partition'],
    config=config['data'],
    tokenizer=Tokenizer,
    isTrainPartition=False,
    global_stats=global_stats
)

test_dataset = ASRDataset(
    partition=config['data']['test_partition'],
    config=config['data'],
    tokenizer=Tokenizer,
    isTrainPartition=False,
    global_stats=global_stats
)

gc.collect()

Loading data for train-clean-100 partition...


100%|██████████| 28539/28539 [01:29<00:00, 318.15it/s]


Global stats computed from training set.
Loading data for dev-clean partition...


100%|██████████| 2703/2703 [00:05<00:00, 539.71it/s]


Loading data for test-clean partition...


100%|██████████| 2620/2620 [00:03<00:00, 810.47it/s]


2046

In [32]:
train_loader    = DataLoader(
    dataset     = train_dataset,
    batch_size  = config['data']['batch_size'],
    shuffle     = True,
    num_workers = config['data']['NUM_WORKERS'] if device == 'cuda' else 0,
    pin_memory  = True,
    collate_fn  = train_dataset.collate_fn
)

val_loader      = DataLoader(
    dataset     = val_dataset,
    batch_size  = config['data']['batch_size'],
    shuffle     = False,
    num_workers = config['data']['NUM_WORKERS'] if device == 'cuda' else 0,
    pin_memory  = True,
    collate_fn  = val_dataset.collate_fn
)

test_loader     = DataLoader(
    dataset     = test_dataset,
    batch_size  = config['data']['batch_size'],
    shuffle     = False,
    num_workers = config['data']['NUM_WORKERS'] if device == 'cuda' else 0,
    pin_memory  = True,
    collate_fn  = test_dataset.collate_fn
)

gc.collect()

0

### Dataloader Verification

In [33]:
verify_dataloader(train_loader)

             Dataloader Verification              
Dataloader Partition     : train-clean-100
--------------------------------------------------
Number of Batches        : 1784
Batch Size               : 16
--------------------------------------------------
Checking shapes of the data...                    

Feature Shape            : [16, 2099, 80]
Shifted Transcript Shape : [16, 76]
Golden Transcript Shape  : [16, 76]
Feature Lengths Shape    : [16]
Transcript Lengths Shape : [16]
--------------------------------------------------
Max Feature Length       : 3066
Max Transcript Length    : 139
Avg. Chars per Token     : 3.13


In [34]:
verify_dataloader(val_loader)

             Dataloader Verification              
Dataloader Partition     : dev-clean
--------------------------------------------------
Number of Batches        : 169
Batch Size               : 16
--------------------------------------------------
Checking shapes of the data...                    

Feature Shape            : [16, 3676, 80]
Shifted Transcript Shape : [16, 132]
Golden Transcript Shape  : [16, 132]
Feature Lengths Shape    : [16]
Transcript Lengths Shape : [16]
--------------------------------------------------
Max Feature Length       : 4081
Max Transcript Length    : 179
Avg. Chars per Token     : 3.08


In [35]:
verify_dataloader(test_loader)

             Dataloader Verification              
Dataloader Partition     : test-clean
--------------------------------------------------
Number of Batches        : 164
Batch Size               : 16
--------------------------------------------------
Checking shapes of the data...                    

Feature Shape            : [16, 1556, 80]
Feature Lengths Shape    : [16]
--------------------------------------------------
Max Feature Length       : 4370
Max Transcript Length    : 0
Avg. Chars per Token     : 0.00


## Calculate Max Lengths
Calculating the maximum transcript length across your dataset is a crucial step when working with certain transformer models.
-  The positional encodings, sinusoidal or rotary, are precomputed up to a fixed maximum length.
- This maximum length is a hyperparameter that determines:
  - How long of a sequence your model can process
  - The size of your positional encoding matrix
  - Memory requirements during training and inference
- `Requirements`: For this assignment, ensure your positional encodings can accommodate at least the longest sequence in your dataset to prevent truncation.
- `NOTE`: We'll be using the same positional encoding matrix for all sequences in your dataset. Take this into account when setting your maximum length.

In [39]:
max_feat_len       = max(train_dataset.feat_max_len, val_dataset.feat_max_len, test_dataset.feat_max_len)
max_transcript_len = max(train_dataset.text_max_len, val_dataset.text_max_len, test_dataset.text_max_len)
max_len            = max(max_feat_len, max_transcript_len)

print("="*50)
print(f"{'Max Feature Length':<30} : {max_feat_len}")
print(f"{'Max Transcript Length':<30} : {max_transcript_len}")
print(f"{'Overall Max Length':<30} : {max_len}")
print("="*50)

Max Feature Length             : 4370
Max Transcript Length          : 179
Overall Max Length             : 4370


## Wandb

In [36]:
import os
import wandb
os.environ["WANDB_ENTITY"] = "hojunj-carnegie-mellon-university"
wandb.login(key="REDACTED")

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hojunj (hojunj-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Training

Every time you run the trainer, it will create a new directory in the `expts` folder with the following structure:
```
expts/
    └── {run_name}/
        ├── config.yaml
        ├── model_arch.txt
        ├── checkpoints/
        │   ├── checkpoint-best-metric-model.pth
        │   └── checkpoint-last-epoch-model.pth
        ├── attn/
        │   └── {attention visualizations}
        └── text/
            └── {generated text outputs}
```


### Training Strategy 1: Cold-Start Trainer

#### Model Load (Default)

In [40]:
model_config = config['model'].copy()
model_config.update({
    'input_dim': config['data']['num_feats'],
    'max_len': max_len,
    'num_classes': Tokenizer.vocab_size
})

model = EncoderDecoderTransformer(**model_config)

# Get some inputs from the train dataloader
for batch in train_loader:
    padded_feats, padded_shifted, padded_golden, feat_lengths, transcript_lengths = batch
    break

total_param = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert total_param < 30_000_000, f"Total trainable parameters ({total_param}) exceeds 30 million."

model_stats = summary(model, input_data=[padded_feats, padded_shifted, feat_lengths, transcript_lengths])
print(model_stats)

Layer (type:depth-idx)                                       Output Shape              Param #
EncoderDecoderTransformer                                    [16, 83, 1000]            --
├─SpeechEmbedding: 1-1                                       [16, 529, 320]            --
│    └─Conv2DSubsampling: 2-1                                [16, 529, 320]            --
│    │    └─Sequential: 3-1                                  [16, 320, 529, 76]        925,120
│    │    └─Linear: 3-2                                      [16, 529, 320]            7,782,720
│    │    └─Dropout: 3-3                                     [16, 529, 320]            --
├─Dropout: 1-2                                               [16, 529, 320]            --
├─ModuleList: 1-3                                            --                        --
│    └─SelfAttentionEncoderLayer: 2-2                        [16, 529, 320]            819,200
│    │    └─SelfAttentionLayer: 3-4                          [16, 529, 320]   

#### Initialize Trainer

If you need to reload the model from a checkpoint, you can do so by calling the `load_checkpoint` method.

```python
checkpoint_path = "path/to/checkpoint.pth"
trainer.load_checkpoint(checkpoint_path)
```


In [41]:
trainer = ASRTrainer(
    model=model,
    tokenizer=Tokenizer,
    config=config,
    run_name="add_run_name",
    config_file="config.yaml",
    device=device
)

Using device: cuda


### Setup Optimizer and Scheduler

You can set your own optimizer and scheduler by setting the class members in the `ASRTrainer` class.
Eg:
```python
trainer.optimizer = optim.AdamW(model.parameters(), lr=config['optimizer']['lr'], weight_decay=config['optimizer']['weight_decay'])
# The trainer steps the scheduler after every optimizer update (every batch by default), so T_max counts updates, not epochs
trainer.scheduler = optim.lr_scheduler.CosineAnnealingLR(trainer.optimizer, T_max=10 * len(train_loader))
```

We also provide a utility function to create your own optimizer and scheduler with the congig and some extra bells and whistles. You are free to use it or not. Do read their code and documentation to understand how it works (`hw3lib/utils/*`).


#### Setting up the optimizer

In [ ]:
trainer.optimizer = create_optimizer(
    model=model,
    opt_config=config['optimizer']
)

#### Creating a test scheduler and plotting the learning rate schedule

In [ ]:
test_scheduler = create_scheduler(
    optimizer=trainer.optimizer,
    scheduler_config=config['scheduler'],
    train_loader=train_loader,
    gradient_accumulation_steps=config['training']['gradient_accumulation_steps']
)

plot_lr_schedule(
    scheduler=test_scheduler,
    num_epochs=20,
    train_loader=train_loader,
    gradient_accumulation_steps=config['training']['gradient_accumulation_steps']
)

#### Setting up the scheduler

In [ ]:
trainer.scheduler = create_scheduler(
    optimizer=trainer.optimizer,
    scheduler_config=config['scheduler'],
    train_loader=train_loader,
    gradient_accumulation_steps=config['training']['gradient_accumulation_steps']
)

#### Train
- Set your epochs and start training!
- `NOTE`: The optimizer and scheduler have to be set on the trainer before this call, as in the cells above.

In [ ]:
run_name="d320_e8d4_24ep",

In [ ]:
import time
start = time.time()
trainer.train(train_loader, val_loader, epochs=24)
print(f"Total training took {(time.time() - start) / 3600:.1f} hours")

#### Inference



##### W&B Run Tracking for Kaggle Submission

The W&B run is tracked automatically when you train and load
checkpoints using the provided trainer.

Normally, leave `KAGGLE_WANDB_RUN_OVERRIDE = None`.

If you manually replace model weights using
`model.load_state_dict(...)` instead of `trainer.load_checkpoint(...)`,
set `KAGGLE_WANDB_RUN_OVERRIDE` to the exact W&B run that produced
those weights. Manual weight loading does not update the model's
stored W&B association.

Format: `"entity/project/run_id"`

The inference cell ends the trainer's W&B run (`trainer.cleanup()`).
To continue training afterward, first rerun the cells from
`trainer = ASRTrainer(...)` through the scheduler setup. The new
trainer starts a new W&B run. Then reset the override to `None`
before generating a new Kaggle submission, so the trainer can use
the run associated with the new training.

In [48]:
# Optional W&B provenance override for this Kaggle submission.
# Normally, leave this as None.
KAGGLE_WANDB_RUN_OVERRIDE = None

In [43]:
import time
for bw in [1, 3]:
    cfg = {'num_batches': 10, 'temperature': 1.0, 'repeat_penalty': 1.0, 'beam_width': bw}
    t0 = time.time()
    res = trainer.recognize(val_loader, cfg, config_name=f'beam{bw}', max_length=max_transcript_len)
    m = trainer._calculate_asr_metrics([r['target'] for r in res], [r['generated'] for r in res])
    print(f"beam_width={bw}: CER={m['cer']:.2f}, WER={m['wer']:.2f}, time={time.time()-t0:.0f}s")

beam_width=1: CER=7.34, WER=15.27, time=17s


beam_width=3: CER=9.89, WER=17.72, time=44s


In [44]:
cfg3 = {'num_batches': 1, 'temperature': 1.0, 'repeat_penalty': 1.0, 'beam_width': 3}
res3 = trainer.recognize(val_loader, cfg3, config_name='beam3_check', max_length=max_transcript_len)
for r in res3[:3]:
    print("TARGET:", r['target'][:150]); print("BEAM3 :", r['generated'][:150]); print()

TARGET: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL
BEAM3 : MISTER CHILDREN IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL

TARGET: NOR IS MISTER QUILTER'S MANNER LESS INTERESTING THAN HIS MATTER
BEAM3 : NOR IS MISTER CULTURES MANNER LESS INTERESTING THAN HIS MATTER

TARGET: HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MO
BEAM3 : HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROSE BEEF LOVING BEFORE US SIMILIES DRAWN FROM EATING IN ITS RESULTS OCCUR MOST



In [45]:
cfg1 = {'num_batches': 10, 'temperature': 1.0, 'repeat_penalty': 1.0, 'beam_width': 1}
cfg3 = {'num_batches': 10, 'temperature': 1.0, 'repeat_penalty': 1.0, 'beam_width': 3}
r1 = trainer.recognize(val_loader, cfg1, config_name='g', max_length=max_transcript_len)
r3 = trainer.recognize(val_loader, cfg3, config_name='b', max_length=max_transcript_len)
diffs = sorted(range(len(r1)), key=lambda i: abs(len(r3[i]['generated']) - len(r1[i]['generated'])), reverse=True)
for i in diffs[:3]:
    print(f"[{i}] target len={len(r1[i]['target'])}, greedy len={len(r1[i]['generated'])}, beam len={len(r3[i]['generated'])}")
    print("  BEAM3 :", r3[i]['generated'][-120:]); print()

[71] target len=88, greedy len=83, beam len=505
  BEAM3 :  FROM IROLLED OUT FROM IROLLED OUT FROM I ROLLED OUT FROM I ROLLED OUT FROM I ROLLED OUT FROM I ROLLED OUT FROM I ROLLED

[72] target len=67, greedy len=92, beam len=70
  BEAM3 : THEN THE POWERFUL TWISTS THAT THRUST TO THE SIDE IN AN UNDER THE GUARD

[4] target len=362, greedy len=213, beam len=219
  BEAM3 : TIONAL AS A JINGLE PALM MISTER BROOKER'S LANDSCAPES SMILE AT ONE MUCH IN THE SAME WAY THAT MISTER KARKER'S BATH NEXT MAN



In [46]:
import time
for bw, rp in [(1, 1.0), (3, 1.1), (3, 1.2), (3, 1.3)]:
    cfg = {'num_batches': 10, 'temperature': 1.0, 'repeat_penalty': rp, 'beam_width': bw}
    t0 = time.time()
    res = trainer.recognize(val_loader, cfg, config_name=f'b{bw}_rp{rp}', max_length=max_transcript_len)
    m = trainer._calculate_asr_metrics([r['target'] for r in res], [r['generated'] for r in res])
    print(f"beam={bw}, rp={rp}: CER={m['cer']:.2f}, time={time.time()-t0:.0f}s")

beam=1, rp=1.0: CER=7.34, time=17s


beam=3, rp=1.1: CER=6.84, time=53s


beam=3, rp=1.2: CER=6.65, time=51s


beam=3, rp=1.3: CER=6.63, time=41s


In [47]:
import time
for bw, rp in [(1, 1.3), (3, 1.5), (5, 1.3), (5, 1.5)]:
    cfg = {'num_batches': 10, 'temperature': 1.0, 'repeat_penalty': rp, 'beam_width': bw}
    t0 = time.time()
    res = trainer.recognize(val_loader, cfg, config_name=f'b{bw}_rp{rp}', max_length=max_transcript_len)
    m = trainer._calculate_asr_metrics([r['target'] for r in res], [r['generated'] for r in res])
    print(f"beam={bw}, rp={rp}: CER={m['cer']:.2f}, time={time.time()-t0:.0f}s")

beam=1, rp=1.3: CER=7.17, time=19s


beam=3, rp=1.5: CER=7.05, time=41s


beam=5, rp=1.3: CER=6.74, time=62s


beam=5, rp=1.5: CER=7.08, time=81s


In [49]:
# DO NOT CHANGE ================================

# Clear results from any previous inference attempt.
KAGGLE_SUBMISSION_MESSAGE = None
results_df = None

# Check provenance before running inference.
KAGGLE_SUBMISSION_MESSAGE = trainer.build_kaggle_submission_message(
    homework_tag="HW3P2",
    manual_wandb_run=KAGGLE_WANDB_RUN_OVERRIDE,
)
# ===============================================
 
# Define the recognition config: Greedy search
recognition_config = {
    'num_batches': None,
    'temperature': 1.0,
    'repeat_penalty': 1.3,
    'beam_width': 3, # Beam width of 1 reverts to greedy
}

# Recognize with the config above
config_name = "test"
print(f"Evaluating with {config_name} config")
results = trainer.recognize(test_loader, recognition_config, config_name=config_name, max_length=max_transcript_len)

# Calculate metrics on full batch
generated = [r['generated'] for r in results]
results_df = pd.DataFrame(
    {
        'id': range(len(generated)),
        'transcription': generated,
    }
)

print("Kaggle submission tracking:")
print(KAGGLE_SUBMISSION_MESSAGE)

# Cleanup (will end the active W&B run. To train again, create a new trainer first)
trainer.cleanup()

Evaluating with test config


Kaggle submission tracking:
HW3P2|WB=hojunj-carnegie-mellon-university/hw3p2/n3ocb7me


In [ ]:
print(KAGGLE_SUBMISSION_MESSAGE)
print(len(results_df))
results_df.head(3)

## Submit to Kaggle

In [ ]:
# DO NOT CHANGE
if (
    globals().get("results_df") is None
    or not globals().get("KAGGLE_SUBMISSION_MESSAGE")
):
    raise RuntimeError("Run the inference cell successfully before submitting.")

results_df.to_csv("results.csv", index=False)

api.competition_submit(
    file_name="results.csv",
    message=KAGGLE_SUBMISSION_MESSAGE,
    competition="f-26-hw-3-p-2-student-competition",
)

In [13]:
import os
os.environ['KAGGLE_USERNAME']  = "hojunjin999"
os.environ['KAGGLE_API_TOKEN'] = "REDACTED"

!kaggle competitions submit -c f-26-hw-3-p-2-student-slack-competition -f /tmp/ckpt/idl-hw3p2/results.csv -m "HW3P2|WB=hojunj-carnegie-mellon-university/hw3p2/n3ocb7me"

100%|█████████████████████████████████████████| 289k/289k [00:00<00:00, 631kB/s]
Successfully submitted to 11785 HW3P2 ASR with Transformer Fall 2026 Slack

In [50]:
print(len(results_df))
results_df.to_csv("results_beam3.csv", index=False)

import os
os.environ['KAGGLE_USERNAME']  = "hojunjin999"
os.environ['KAGGLE_API_TOKEN'] = "KGAT_put_your_token_here"

!kaggle competitions submit -c f-26-hw-3-p-2-student-slack-competition -f results_beam3.csv -m "HW3P2|WB=hojunj-carnegie-mellon-university/hw3p2/n3ocb7me"

2620
100%|█████████████████████████████████████████| 286k/286k [00:00<00:00, 712kB/s]
Successfully submitted to 11785 HW3P2 ASR with Transformer Fall 2026 Slack

##### View the associated W&B run

After submitting, open the competition's **Submissions** tab.
The description beneath each submission shows its associated W&B run:

`HW3P2|WB=entity/project/run_id`

To open that run in W&B, use:

`https://wandb.ai/entity/project/runs/run_id`

This lets you identify which experiment produced each Kaggle submission.


#### **Manual submission (optional)**

If automatic submission is unavailable, download the CSV and submit it manually using one of the options below.

When submitting manually, copy the exact **KAGGLE_SUBMISSION_MESSAGE** printed above into the Kaggle submission description. This information is required to associate the submission with the W&B experiment used to generate it.

##### **Kaggle**

* After running the cell above, the file **`results.csv`** will be saved in:
  **Right sidebar → Output → `kaggle/working/<your repo>/`**
* If the file does not appear right away, click the **refresh icon** in the top-right corner of the Output panel.
* Once the file is visible, **right-click → Download**.
* Upload the downloaded file to the **Kaggle competition submission page**.

---

##### **Colab**

```python
from google.colab import files
files.download("results.csv")
```

##### **PSC**

* Open the left file browser.
* Navigate to:
  `/jet/home/<your_username>/<your repo>/`
* Locate **`results.csv`**.
* Right-click the file and select **Download**.
* If the file does not appear immediately, refresh the file browser.

## 📦 Gradescope Submission Instructions (Read Carefully)

This section guides you through creating and submitting your **final Gradescope code submission**.
Please follow each step in order. Skipping steps may result in an incomplete or invalid submission.

### ⏰ Important Deadlines

* **Kaggle Final Deadline:** See the course schedule
* **Gradescope Code Submission Deadline:**
  **48 hours after** the Kaggle deadline (or the same day as your Slack Kaggle submission, if applicable)

You must submit **both**:

1. A valid Kaggle submission
2. A complete Gradescope code submission zip

Please assign your **final trained model** to the global variable `MODEL`.

We will use this variable to automatically generate a file named
`model_metadata.json`, which records important information about your model (such as parameter count and architecture).

#### 🔴 Important Requirements

* **`MODEL` must be the exact model used for your best Kaggle submission**
* This step is **required** for grading and audit purposes
* If the model does not match your Kaggle submission, your score may be invalidated

#### 🕒 When to Do This

* Run this cell **only after you have finished training** your final model
* Do **not** retrain or modify the model after assigning it to `MODEL`


### Step 1: Confirm Your Final Model

Keep `MODEL = trainer.model` when the current trainer contains
the model used for your best Kaggle submission.

The final submission cell prints the best scored Kaggle submission
and its W&B run. If that submission used a different model,
load its checkpoint into the trainer and rerun the MODEL cell
before recreating the ZIP.

Normally, no change is needed. Changing MODEL does not change
which W&B runs are selected.

In [ ]:
MODEL = trainer.model 

### Step 2: Complete the README Section

Fill in the `README` variable with a brief description of your work:

* **Model**: Architecture and key design choices
* **Training Strategy**: Optimizer, scheduler, loss, etc.
* **Augmentations**: If used (omit if not)
* **Notebook Execution**: Any instructions needed to run your notebook

This README will be included in your submission zip.

In [ ]:
README = """
- **Model**: Model archtiecture description. Anything unique? Any specific architecture shapes or strategies?
- **Training Strategy**: optimizer + scheduler + loss function + any other unique ideas
- **Augmentations**: augmentations if used. If augmentations weren't used, then ignore
- **Notebook Execution**: Any instructions required to run your notebook.
"""

### Step 3: Provide Required Credentials

Ensure the following variables are correctly set:

* `KAGGLE_USERNAME`
* `KAGGLE_API_KEY`: the same `KGAT_` token you used to log in to Kaggle
* `WANDB_API_KEY`
* `WANDB_USERNAME_OR_TEAMNAME`
* `WANDB_PROJECT`

These are used **only** to:

* Fetch your official Kaggle score
* Retrieve top W&B runs associated with your Kaggle submissions for grading and audit purposes.

By default, the backend selects up to 10 distinct W&B runs according to the homework's Kaggle score ordering. You may optionally customize the selected list using **FINAL_WANDB_RUNS_OVERRIDE** (see the next cell).

⚠️ Do **not** share your API keys publicly.

In [ ]:
# @title
KAGGLE_USERNAME = "<your-username>" # TODO
KAGGLE_API_KEY  = "<your-key>"      # TODO: the same API token that starts with KGAT_
WANDB_API_KEY   = "<ENTER-YOUR-WANDB-API-KEY-HERE>" # TODO
WANDB_USERNAME_OR_TEAMNAME = "<ENTER-YOUR-WANDB-USERNAME-OR-TEAMNAME-HERE>" # TODO: Put your username-or-team-name here
WANDB_PROJECT              = "<ENTER-YOUR-WANDB-PROJECT-NAME-HERE>" # TODO: Put your project-name

##### **Optional W&B Run Selection Override**

By default, the submission backend automatically selects up to **10 distinct W&B runs** associated with your eligible Kaggle submissions, ordered according to the official Kaggle metric for this homework.

**In normal use, leave `FINAL_WANDB_RUNS_OVERRIDE = []`.**

Only modify this list if you intentionally want to change the automatically selected W&B runs after reviewing the selection printed by the final submission cell.

Each entry must use the exact format:

`"entity/project/run_id"`

For example:

```python
FINAL_WANDB_RUNS_OVERRIDE = [
    "my-team/HW3P2-F26/abc123xyz",
    "my-team/HW3P2-F26/def456uvw",
]
```
**Requirements:**
- You may specify at most 10 runs.
- Every run must be associated with one of your eligible Kaggle submissions.
- The run associated with your top-ranked Kaggle submission must remain included.
- Leave the list empty to use the automatic selection.

In [ ]:
# Optional:
# Leave this list empty to use the W&B runs selected automatically
# from your eligible Kaggle submissions.
#
# If overriding, provide at most 10 exact W&B run paths using:
#     "entity/project/run_id"
#
# Example:
# FINAL_WANDB_RUNS_OVERRIDE = [
#     "my-team/HW3P2-F26/abc123xyz",
#     "my-team/HW3P2-F26/def456uvw",
# ]
FINAL_WANDB_RUNS_OVERRIDE = []

### Step 4: Set File Paths Correctly

You must provide absolute path to your **final notebook**

### Platform-specific guidance

**Colab**

* Right-click a file in the left file pane → **Copy path**
* Paths typically start with `/content/...`

**Kaggle**

* Download your notebook (`File → Download Notebook`)
* Upload it via **Upload Input → Upload Model**
* Copy paths from the right sidebar (`/kaggle/working/...`)

**PSC / Linux**

* Locate files under `/jet/home/<your_username>/`
* Use `!ls` to confirm paths

Paste the correct path below.

In [ ]:
NOTEBOOK_PATH = "" # TODO

### Step 5: (Optional) Additional Files

If you have extra files you want to include (e.g., config files, scripts), add their paths to `ADDITIONAL_FILES`, Otherwise, leave this list empty.

In [ ]:
ADDITIONAL_FILES = []

### Step 6: Generate the Final Submission Zip

Before running the submission cell:

* Ensure `ACKNOWLEDGED = True`
* Double-check all paths and credentials
* Confirm your Kaggle submission exists

Then run the cell below.
This should generate a `HW3P2_final_submission.zip` which includes:
* Your notebook
* `model_metadata.json`
* README
* WandB run exports
* Kaggle metadata
* Academic integrity acknowledgement

In [ ]:
#### DO NOT MODIFY ####
!git clone https://github.com/CMU-IDeeL/HWP2-Submission-Backend-Student-New.git
!mv HWP2-Submission-Backend-Student-New/submission .
!rm -rf HWP2-Submission-Backend-Student-New
from submission.submission_config import SubmissionConfig
from submission.backend_config import BackendConfig, HW3P2_BACKEND_CONFIG
from submission.main import create_submission_zip

create_submission_zip(
    cfg = SubmissionConfig(
        model = MODEL,
        kaggle_username     = KAGGLE_USERNAME,
        kaggle_api_key      = KAGGLE_API_KEY,
        wandb_api_key       = WANDB_API_KEY,
        wandb_entity        = WANDB_USERNAME_OR_TEAMNAME,
        wandb_project       = WANDB_PROJECT,
        wandb_runs_override = FINAL_WANDB_RUNS_OVERRIDE,
        acknowledged        = ACKNOWLEDGED,
        readme              = README,
        notebook_path       = NOTEBOOK_PATH,
        additional_files    = ADDITIONAL_FILES
    ),
    backend_cfg = HW3P2_BACKEND_CONFIG
)

### Step 7: Upload to Gradescope

1. Download `HW3P2_final_submission.zip`
2. Upload it to **Gradescope**
3. Verify the upload succeeds

⚠️ Submitting an incomplete zip or modifying backend code may result in grading penalties or an Academic Integrity Violation.

---

### ✅ Final Checklist

Before submitting, confirm:

* [ ] Kaggle username is valid
* [ ] `model_metadata.json` matches your best run
* [ ] README is complete
* [ ] File paths are correct
* [ ] `ACKNOWLEDGED = True`
* [ ] Final zip downloads successfully